# 🧪 Exploration des données (Couche Bronze)

Ce notebook permet de visualiser et d'explorer les données ingérées dans la couche Bronze. 
**Objectif :** Vérifier la qualité des données, identifier les anomalies (valeurs nulles, doublons) avant de passer à la couche Silver.

In [ ]:
import os
from src.common.spark_session_manager import get_spark_session
from src.config import NIVEAU_VIE_PAUVRETE_BRONZE_PATH
from pyspark.sql.functions import col

# 1. Initialiser la session Spark
spark = get_spark_session(app_name="Exploration_Bronze")

# 2. Configuration "Look Databricks" (Eager Evaluation)
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)

## 📊 Exploration de Niveau de vie et pauvreté 2017 Lyon au km carroyé
Lecture des données depuis le format **Parquet** (stockage optimisé).

In [ ]:
nvp_path = os.path.join(NIVEAU_VIE_PAUVRETE_BRONZE_PATH, "2017_Filosofi2017_carreaux_1km_met")
df_nvp_2017 = spark.read.parquet(nvp_path)

# Aperçu des données
df_nvp_2017

## 🔬 Analyse Technique des Schémas (Data Profiling)
Il est crucial de vérifier les types détectés par Spark pour anticiper le nettoyage en couche Silver.

In [ ]:
# Affichage des types de colonnes (Senior Approach)
print("--- Analyse des types de données ---")
for col_name, dtype in df_nvp_2017.dtypes:
    if "string" in dtype:
        print(f"⚠️ Colonne TEXTE (potentiel nettoyage requis) : {col_name}")
    else:
        print(f"✅ Colonne NUMÉRIQUE : {col_name}")

## 🔍 Statistiques Descriptives
On se concentre sur les colonnes de participation pour vérifier la cohérence des chiffres

In [ ]:
numeric_cols = ["Ind", "Men_1ind", "Men_5ind", "Men_prop", "Men_fmp", "Ind_snv", "Men_surf",
                "Men_coll", "Men_mais", "Log_av45", "Log_45_70", "Log_70_90", "Log_ap90", "Log_inc",
                "Log_soc", "Ind_0_3", "Ind_4_5", "Ind_6_10", "Ind_11_17", "Ind_18_24", "Ind_25_39",
                "Ind_40_54", "Ind_55_64", "Ind_65_79", "Ind_80p", "Ind_inc","Men_pauv",	"Men"]
df_nvp_2017.select(numeric_cols).describe().toPandas()

In [ ]:
df_nvp_2017.printSchema()